In [ ]:
import httpx
import json
import base64
import time
import os
import hashlib
from typing import Dict, Any

# --- Configuración del Cliente ---
BASE_URL = "http://127.0.0.1:8000"
CLIENT_HTTP = httpx.Client(base_url=BASE_URL, timeout=10.0)

# --- Variables Globales de Estado ---
SESSION_ID = None
SERVER_PUBLIC_ELEMENT = None 

# --- HASH SIMULADO (Debe coincidir con el servidor) ---
HASH_FUNC = hashlib.sha256

# --- Generación Determinista de Placeholders ---

def calculate_simulated_proof(element: str) -> str:
    """
    Calcula la prueba simulada que el servidor acepta (M1).
    
    IMPORTANTE: Usa .hexdigest() para garantizar que la prueba esté en formato
    hexadecimal (64 caracteres), coincidiendo con la expectativa del servidor.
    """
    hash_input = f"proof:{element}"
    return HASH_FUNC(hash_input.encode()).hexdigest()

# --- Datos Fijos del Cliente (Placeholders) ---
USER_DATA = {
    "username": "notebook_alice",
    "salt": base64.b64encode(os.urandom(16)).decode('utf-8'),
    "verifier": base64.b64encode(os.urandom(32)).decode('utf-8')
}
# Elemento público X_A (simulado como Base64)
CLIENT_ELEMENT_A = base64.b64encode(os.urandom(64)).decode('utf-8')

# M1 se calcula usando la función corregida, asegurando que es HEX
CLIENT_PROOF_M1 = calculate_simulated_proof(CLIENT_ELEMENT_A) 

print(f"URL Base del Servidor: {BASE_URL}")
print(f"Usuario de Prueba: {USER_DATA['username']}")
print(f"Prueba M1 calculada (HEX): {CLIENT_PROOF_M1}")
print(f"Longitud de M1: {len(CLIENT_PROOF_M1)}")

URL Base del Servidor: http://127.0.0.1:8000
Usuario de Prueba: notebook_alice
Prueba M1 calculada (HEX): 5120d25c4c44578cfc0929230544415a15dd123e2e2fb4c330ce0db9497cdce9
Longitud de M1: 64


In [13]:
def print_separator(title):
    """Imprime un separador bonito para las etapas del flujo."""
    print("\n" + "="*80)
    print(f"| {title.upper()}")
    print("="*80)

def handle_response(response: httpx.Response, step_name: str, expected_status: int = 200) -> Dict[str, Any]:
    """Maneja la respuesta HTTP y verifica el código de estado."""
    print(f"Petición a: {response.request.url}")
    print(f"Código de Estado: {response.status_code}")
    
    try:
        response_json = response.json()
    except json.JSONDecodeError:
        response_json = {"error": "Respuesta no JSON", "text": response.text}

    if response.status_code == expected_status:
        print(f"[OK] {step_name} completado exitosamente.")
        return response_json
    else:
        print(f"[ERROR CRÍTICO] {step_name} falló. Esperado {expected_status}, Obtenido {response.status_code}.")
        print("Detalles del Error:", response_json)
        # En un notebook, lanzamos una excepción para detener el flujo en lugar de sys.exit()
        raise Exception(f"Fallo en el paso {step_name}")

In [14]:
print_separator("Paso 1: Registro de Usuario (Setup)")

register_request_data = {
    "username": USER_DATA["username"],
    "salt": USER_DATA["salt"],
    "verifier": USER_DATA["verifier"]
}

print("Datos de Registro:", register_request_data)

try:
    response = CLIENT_HTTP.post("/register", json=register_request_data)
    
    # Debe esperar 201 Created
    result = handle_response(response, "Registro de Usuario", 201)
    print("\nRespuesta del Servidor:", result)

except Exception as e:
    print(f"\n[FALLO] No se pudo conectar o registrar: {e}")


| PASO 1: REGISTRO DE USUARIO (SETUP)
Datos de Registro: {'username': 'notebook_alice', 'salt': '29PeMXFPcu1gioBY42/OiQ==', 'verifier': '/DoAzXcs8Ji/hHKutTwWU/LdTRoZ0eZCM5pMJpITsgk='}
Petición a: http://127.0.0.1:8000/register
Código de Estado: 201
[OK] Registro de Usuario completado exitosamente.

Respuesta del Servidor: {'message': 'Registro de usuario exitoso.'}


In [15]:
print_separator("Paso 2: Inicio del PAKE (Client -> Server)")

pake_start_data = {
    "username": USER_DATA["username"],
    "client_public_element": CLIENT_ELEMENT_A
}

print("Datos de Inicio PAKE:", pake_start_data)

try:
    response = CLIENT_HTTP.post("/pake/start", json=pake_start_data)
    
    # Debe esperar 200 OK
    start_response = handle_response(response, "Inicio del PAKE")
    
    # --- Almacenamiento del Estado Global ---
    global SESSION_ID, SERVER_PUBLIC_ELEMENT
    SESSION_ID = start_response.get("session_id")
    SERVER_PUBLIC_ELEMENT = start_response.get("server_public_element")
    
    if not SESSION_ID or not SERVER_PUBLIC_ELEMENT:
        raise ValueError("Respuesta de inicio PAKE incompleta: faltan ID o elemento público.")

    print(f"\n[INFO] Session ID recibido (GLOBAL): {SESSION_ID}")
    print(f"[INFO] Elemento Público Servidor (X_B): {SERVER_PUBLIC_ELEMENT}")
    
except Exception as e:
    print(f"\n[FALLO] Error en el inicio del PAKE: {e}")


| PASO 2: INICIO DEL PAKE (CLIENT -> SERVER)
Datos de Inicio PAKE: {'username': 'notebook_alice', 'client_public_element': '6HU5oIi30KnMnNYjF4PjhVawaVbvGWz3QeAl5mlTezA1/TkCcBMu9bqBcdPMW4xdGeSysaOhohO49SA4W2NyjA=='}
Petición a: http://127.0.0.1:8000/pake/start
Código de Estado: 200
[OK] Inicio del PAKE completado exitosamente.

[INFO] Session ID recibido (GLOBAL): a57f4826-08f1-40d4-b0a2-3b76d557ac56
[INFO] Elemento Público Servidor (X_B): e1d3d867d079e0e8ef619f34c2c4ed7ed7c698159b316d322078d45aef6c0479


In [16]:
print_separator("Paso 3: Completar PAKE (Client -> Server)")

global SESSION_ID

if not SESSION_ID:
    print("[ERROR] El ID de sesión no se estableció en el paso anterior. No se puede continuar.")
else:
    pake_complete_data = {
        "client_proof": CLIENT_PROOF_M1
    }
    
    print(f"ID de Sesión usado: {SESSION_ID}")
    print("Datos de Compleción PAKE (Prueba M1):", pake_complete_data)

    try:
        response = CLIENT_HTTP.post(f"/pake/complete/{SESSION_ID}", json=pake_complete_data)
        
        # Debe esperar 200 OK
        complete_response = handle_response(response, "Completar PAKE")
        
        session_key = complete_response.get("session_key_derived")
        server_proof = complete_response.get("server_proof")
        
        print("\n" + "#"*50)
        print(f"[RESUMEN] INTERCAMBIO PAKE EXITOSO.")
        print(f"Clave de Sesión Derivada (SK): {session_key[:30]}...")
        print(f"Prueba del Servidor (M2): {server_proof[:30]}...")
        print("#"*50)

    except Exception as e:
        print(f"\n[FALLO] Error al completar el PAKE: {e}")


| PASO 3: COMPLETAR PAKE (CLIENT -> SERVER)
ID de Sesión usado: a57f4826-08f1-40d4-b0a2-3b76d557ac56
Datos de Compleción PAKE (Prueba M1): {'client_proof': '5120d25c4c44578cfc0929230544415a15dd123e2e2fb4c330ce0db9497cdce9'}
Petición a: http://127.0.0.1:8000/pake/complete/a57f4826-08f1-40d4-b0a2-3b76d557ac56
Código de Estado: 200
[OK] Completar PAKE completado exitosamente.

##################################################
[RESUMEN] INTERCAMBIO PAKE EXITOSO.
Clave de Sesión Derivada (SK): 4220131c9dd63985261657ef027306...
Prueba del Servidor (M2): 3bdd6a53574d79cb2b6a3c23b366b0...
##################################################
